# Falcon Perception — Interactive Notebook

This notebook demonstrates **Falcon Perception**, an early-fusion, natively multimodal, dense autoregressive Transformer for open-vocabulary segmentation and detection model.

In this notebook, we walk through:
1. **Getting Started** — loading the model, building the inference engine, and a warmup run
2. **Features** — detection mode, HR feature cache, and dense segmentation parameter tuning
3. **Level Showcase** — progressively harder queries on a single image (PBench levels 0–4)
4. **Benchmark** — quantitative evaluation on PBench with IoU metrics


### Prerequisites

- A CUDA-capable GPU (tested on A100 80 GB / H100)
- `falcon-perception` package installed (`pip install -e .` from this repo)

In [ ]:
import torch
from PIL import Image

from falcon_perception import (
    PERCEPTION_MODEL_ID,
    build_prompt_for_task,
    cuda_timed,
    load_and_prepare_model,
    setup_torch_config,
)
from falcon_perception.data import ImageProcessor, stream_samples_from_hf_dataset
from falcon_perception.paged_inference import (
    PagedInferenceEngine,
    SamplingParams,
    Sequence,
    engine_config_for_gpu,
)
from falcon_perception.visualization_utils import render_sequence_overlay

setup_torch_config()


### Key Parameters

- **`max_seq_length`** — The model was trained with sequences of up to **8192** tokens, so we hard-code it here. Depending on your use case (small images, few objects per query), you can lower it.
- **`min_image_size`** / **`max_image_size`** — Control how images are resized before tokenisation. Larger images produce more vision tokens (and better spatial resolution), but consume more of the sequence budget.

The model can handle arbitrary aspect ratio and resolution. We set these as constants as it was trained with for the rest of the notebook.


In [ ]:
MAX_SEQ_LENGTH = 8192
MIN_IMAGE_SIZE = 256
MAX_IMAGE_SIZE = 1024


### Load Model

Download the Falcon Perception checkpoint from Hugging Face and prepare it for inference.
`load_and_prepare_model` handles downloading, dtype casting, and `torch.compile`.


In [ ]:
model, tokenizer, model_args = load_and_prepare_model(
    hf_model_id=PERCEPTION_MODEL_ID,
    dtype="float32",
    compile=True,
)

image_processor = ImageProcessor(patch_size=16, merge_size=1)
stop_token_ids = [tokenizer.eos_token_id, tokenizer.end_of_query_token_id]


### Build the Paged Inference Engine

`engine_config_for_gpu()` auto-detects your GPU's VRAM and returns recommended settings
(batch size, number of KV-cache pages, prefill length limit).

We enable **`enable_hr_cache=True`** so that the expensive high-resolution upsampler
features are cached in CPU-pinned memory and reused across queries on the same image.


In [ ]:
cfg = engine_config_for_gpu(max_image_size=MAX_IMAGE_SIZE, dtype=model.dtype)
print(f"Auto-config: {cfg}")

engine = PagedInferenceEngine(
    model, tokenizer, image_processor,
    max_seq_length=MAX_SEQ_LENGTH,
    enable_hr_cache=True,
    capture_cudagraph=True,
    **cfg,
)


### Warmup

The first run absorbs `torch.compile` JIT and CUDA graph capture costs.
Subsequent runs are much faster.


In [ ]:
warmup_sample = stream_samples_from_hf_dataset("tiiuae/PBench", split="level_1", limit=1)[0]
warmup_prompt = build_prompt_for_task(str(warmup_sample["expression"]), "segmentation")
warmup_seq = Sequence(
    text=warmup_prompt,
    image=warmup_sample["image"],
    min_image_size=MIN_IMAGE_SIZE,
    max_image_size=MAX_IMAGE_SIZE,
)
sp = SamplingParams(stop_token_ids=stop_token_ids)

print("Warmup run ...")
with cuda_timed(reset_peak_memory=False) as warmup_timer:
    engine.generate([warmup_seq], sampling_params=sp, use_tqdm=False, print_stats=False)
print(f"Warmup done in {warmup_timer.elapsed:.1f}s")


---

## Features

In this section we explore key engine capabilities:
- **Detection mode** — skip the HR upsampler when only bounding boxes are needed
- **HR feature cache** — reuse upsampled features across queries on the same image
- **Dense segmentation tuning** — `coord_dedup_threshold` and `hr_upsample_ratio`


In [ ]:
from falcon_perception.data import load_image

DEMO_IMAGE_PATH = "https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/pexels-tahaasamett-10540813.jpg"
demo_img = load_image(DEMO_IMAGE_PATH)
print(f"Demo image: {DEMO_IMAGE_PATH}  size={demo_img.size}")
display(demo_img)


def cap_display(img, max_side=1024):
    """Downscale a PIL image so the longest side is at most max_side, preserving aspect ratio."""
    w, h = img.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)))
    return img


def run_and_show(
    image,
    queries: list[str],
    task: str = "segmentation",
    max_display_side: int = 1024,
):
    """Run inference on an image with the given queries and display overlays."""
    sequences = []
    for i, query in enumerate(queries):
        prompt = build_prompt_for_task(str(query), task)
        sequences.append(Sequence(
            text=prompt,
            image=image,
            min_image_size=MIN_IMAGE_SIZE,
            max_image_size=MAX_IMAGE_SIZE,
            request_idx=i,
            task=task,
        ))

    sp = SamplingParams(stop_token_ids=stop_token_ids)
    with cuda_timed() as t:
        engine.generate(sequences, sampling_params=sp, use_tqdm=True, print_stats=False)

    print(f"\n{len(sequences)} queries in {t.elapsed:.2f}s "
          f"({len(sequences)/t.elapsed:.1f} q/s)")

    for seq, query in zip(sequences, queries):
        n_masks = len(seq.output_aux.masks_rle)
        overlay = render_sequence_overlay(seq, image_processor, draw_bbox=True)
        if overlay is not None:
            print(f"\nQuery: {query!r}  |  Masks: {n_masks}")
            display(cap_display(Image.fromarray(overlay), max_display_side))

    return sequences


### Detection Mode — Skipping the Upsampler

Falcon Perception supports two tasks:

| Task | Output | Upsampler |
|------|--------|-----------|
| `segmentation` | Bounding boxes + high-res binary masks | **Yes** — runs the HR upsampler per mask |
| `detection` | Bounding boxes only | **No** — skips the upsampler entirely |

When you only need bounding boxes (e.g. for counting or coarse localisation), `detection` mode saves the upsampler cost during prefill/finalize, leading to noticeably faster throughput.

Let's compare the two modes on the same image.


In [ ]:
from falcon_perception.visualization_utils import pair_bbox_entries

demo_query = "box"
sp = SamplingParams(stop_token_ids=stop_token_ids)

seg_seq = Sequence(
    text=build_prompt_for_task(demo_query, "segmentation"),
    image=demo_img, min_image_size=MIN_IMAGE_SIZE, max_image_size=MAX_IMAGE_SIZE,
    task="segmentation",
)
engine.generate([seg_seq], sampling_params=sp, use_tqdm=False, print_stats=False)

det_seq = Sequence(
    text=build_prompt_for_task(demo_query, "detection"),
    image=demo_img, min_image_size=MIN_IMAGE_SIZE, max_image_size=MAX_IMAGE_SIZE,
    task="detection",
)
engine.generate([det_seq], sampling_params=sp, use_tqdm=False, print_stats=False)

print(f"Pipeline breakdown for query: {demo_query!r}\n")
header = f"{'Task':>15s}  {'Prefill':>10s}  {'Decode':>10s}  {'Steps':>6s}  {'Finalize':>10s}  {'Masks':>6s}  {'Bboxes':>7s}"
print(header)
print("-" * len(header))
for label, seq in [("Segmentation", seg_seq), ("Detection", det_seq)]:
    n_masks = len(seq.output_aux.masks_rle)
    n_bboxes = len(pair_bbox_entries(seq.output_aux.bboxes_raw))
    print(f"{label:>15s}  {seq.stats.prefill_ms:9.0f}ms  {seq.stats.decode_wall_ms:9.0f}ms  "
          f"{seq.stats.decode_steps:6d}  {seq.stats.finalize_ms:9.0f}ms  {n_masks:6d}  {n_bboxes:7d}")

print("\nDetection skips the HR upsampler, so prefill is faster.")


In [ ]:
from falcon_perception.visualization_utils import overlay_detections_on_image_v2

overlay_seg = render_sequence_overlay(seg_seq, image_processor, draw_bbox=True)

print("Segmentation output (masks + bboxes):")
if overlay_seg is not None:
    display(cap_display(Image.fromarray(overlay_seg)))

# For detection, render bboxes on the original image
bboxes = pair_bbox_entries(det_seq.output_aux.bboxes_raw)
if bboxes:
    det_dets = [
        {"xy": {"x": b["x"], "y": b["y"]}, "hw": {"w": b["w"], "h": b["h"]}}
        for b in bboxes
    ]
    overlay_det = overlay_detections_on_image_v2(
        demo_img, det_dets, draw_bbox=True, masks_are_binary=True,
    )
    print(f"\nDetection output ({len(bboxes)} bboxes):")
    display(cap_display(Image.fromarray(overlay_det)))
else:
    print("\nDetection: No bounding boxes found.")


### HR Feature Cache — Reuse Across Queries

When you ask multiple segmentation queries about the **same image** (e.g. *"the dog"*, then *"the cat"* on the same photo), the expensive high-resolution image feature upsampling only needs to happen once.

Our engine was created with `enable_hr_cache=True`, which stores upsampled HR features in a **CPU-pinned LRU cache** keyed by image hash. On subsequent queries for the same image:
- The upsampler is **skipped** entirely during prefill.
- Cached features are loaded back via async DMA, overlapping with GPU compute.

**CPU RAM cost.** Each cache entry holds the upsampled feature map in pinned memory. For a 1024×1024 image at float32 with `segm_out_dim=256` and `hr_upsample_ratio=8`, one entry is roughly `256 × 512 × 512 × 4 bytes ≈ 256 MiB`. The `max_hr_cache_entries` parameter (auto-set by `engine_config_for_gpu()`) controls how many entries live in the cache. If you're memory-constrained on the host, lower it; if you're running many queries over a fixed set of images, raise it.

Let's see the difference by running two queries on the same image and comparing per-request `prefill_ms` (which includes the upsampler on a cache miss).


In [ ]:
hr_cache_img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/nyt.jpg")
query_a = "photograph"
query_b = "photograph"

prompt_a = build_prompt_for_task(query_a, "segmentation")
prompt_b = build_prompt_for_task(query_b, "segmentation")
sp = SamplingParams(stop_token_ids=stop_token_ids)

print(f"Image: data/nyt.png  size={hr_cache_img.size}")
print(f"Query: {query_a!r}")
print(f"(Running the same query twice on the same image)\n")

# First query — cache miss: upsampler runs during prefill
seq_a = Sequence(text=prompt_a, image=hr_cache_img, min_image_size=MIN_IMAGE_SIZE, max_image_size=MAX_IMAGE_SIZE, task="segmentation")
engine.generate([seq_a], sampling_params=sp, use_tqdm=False, print_stats=False)
print(f"Run 1 (cache MISS): prefill {seq_a.stats.prefill_ms:.0f} ms  |  hr_cache_hit={seq_a._hr_cache_hit}  |  Masks: {len(seq_a.output_aux.masks_rle)}")

# Second query — same image → cache hit: upsampler skipped
seq_b = Sequence(text=prompt_b, image=hr_cache_img, min_image_size=MIN_IMAGE_SIZE, max_image_size=MAX_IMAGE_SIZE, task="segmentation")
engine.generate([seq_b], sampling_params=sp, use_tqdm=False, print_stats=False)
print(f"Run 2 (cache HIT) : prefill {seq_b.stats.prefill_ms:.0f} ms  |  hr_cache_hit={seq_b._hr_cache_hit}  |  Masks: {len(seq_b.output_aux.masks_rle)}")

saved_ms = seq_a.stats.prefill_ms - seq_b.stats.prefill_ms
if seq_a.stats.prefill_ms > 0:
    print(f"\nPrefill time saved: {saved_ms:.0f} ms  ({saved_ms / seq_a.stats.prefill_ms * 100:.0f}% faster on cache hit)")

overlay_a = render_sequence_overlay(seq_a, image_processor, draw_bbox=True)
if overlay_a is not None:
    display(cap_display(Image.fromarray(overlay_a)))


### Dense Segmentation — Many Small Objects

When segmenting scenes with many small objects, two sampling parameters become important:

| Parameter | What it does |
|-----------|-------------|
| `coord_dedup_threshold` | Merges near-duplicate bounding box predictions whose centres are within this normalised distance. Set to `0` to keep all predictions; `0.01` removes redundant overlapping boxes. |
| `hr_upsample_ratio` | Controls the resolution of the HR upsampler output. Higher values (e.g. `16`) produce finer masks — especially helpful for small objects — at the cost of more VRAM and compute. |

We compare two configurations on the same image:

| | Config A (baseline) | Config B (enhanced) |
|---|---|---|
| `coord_dedup_threshold` | 0 | 0.01 |
| `hr_upsample_ratio` | 8 | 16 |

Config B's higher upsample ratio helps resolve small objects, and dedup cleans up redundant predictions.


In [ ]:
dense_image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/car2.jpg")
dense_query = "car"

print(f"Dense image: data/sheep.jpg  size={dense_image.size}")
print(f"Dense query: {dense_query!r}")
display(cap_display(dense_image))


def run_dense_config(image, query, coord_dedup_threshold, hr_upsample_ratio, label=""):
    """Run a single dense segmentation config and return the sequence."""
    prompt = build_prompt_for_task(query, "segmentation")
    seq = Sequence(
        text=prompt,
        image=image,
        min_image_size=MIN_IMAGE_SIZE,
        max_image_size=MAX_IMAGE_SIZE,
        task="segmentation",
    )
    sp = SamplingParams(
        stop_token_ids=stop_token_ids,
        coord_dedup_threshold=coord_dedup_threshold,
        hr_upsample_ratio=hr_upsample_ratio,
    )
    with cuda_timed() as t:
        engine.generate([seq], sampling_params=sp, use_tqdm=False, print_stats=False)

    n_masks = len(seq.output_aux.masks_rle)
    n_bboxes = len(pair_bbox_entries(seq.output_aux.bboxes_raw))
    print(f"  {label}: {n_masks} masks, {n_bboxes} bboxes | "
          f"prefill {seq.stats.prefill_ms:.0f}ms  decode {seq.stats.decode_wall_ms:.0f}ms  "
          f"finalize {seq.stats.finalize_ms:.0f}ms  total {t.elapsed:.2f}s")
    return seq



In [ ]:
print("\nConfig A: coord_dedup_threshold=0, hr_upsample_ratio=8")
seq_config_a = run_dense_config(dense_image, dense_query, coord_dedup_threshold=0, hr_upsample_ratio=8, label="Config A")

print("Config B: coord_dedup_threshold=0.01, hr_upsample_ratio=16")
seq_config_b = run_dense_config(dense_image, dense_query, coord_dedup_threshold=0.01, hr_upsample_ratio=16, label="Config B")

overlay_config_a = render_sequence_overlay(seq_config_a, image_processor, draw_bbox=True)
overlay_config_b = render_sequence_overlay(seq_config_b, image_processor, draw_bbox=True)

n_a = len(seq_config_a.output_aux.masks_rle)
n_b = len(seq_config_b.output_aux.masks_rle)

print(f"Config A (dedup=0, hr=8):   {n_a} masks")
if overlay_config_a is not None:
    display(cap_display(Image.fromarray(overlay_config_a)))

print(f"\nConfig B (dedup=0.01, hr=16): {n_b} masks")
if overlay_config_b is not None:
    display(cap_display(Image.fromarray(overlay_config_b)))


---

## Level Showcase

PBench organises queries into difficulty levels. We demonstrate each level on a single image,
progressing from simple object names to complex compositional expressions.


### Level 0 — Object grounding

Single common nouns. The easiest queries — just name the object.


In [ ]:
run_and_show(demo_img, ["box", "plant"])


### Level 1 — Referring with Attributes

Add adjectives, colours, or simple spatial qualifiers to disambiguate.


In [ ]:
run_and_show(demo_img, ["purple box", "yellow box"])

### Level 2 — OCR-based attribute binding

Filter objects based on its visible text.

In [ ]:
run_and_show(demo_img, ["ACME box"])


### Level 3 — Spatial / Relational Reasoning

Queries that require understanding spatial relationships globally or relative between objects.


In [ ]:
run_and_show(demo_img, ["bottle to the right", "bottle to the left"])


### Level 4 — Compositional / Hardest

Long, compositional expressions that combine attributes, spatial relations, and specific text/labels.


In [ ]:
run_and_show(demo_img, [
    "box with leaf figure",
    "box with boat figure",
    "box in the back row",
])


---

## Benchmark

We run the engine on a subset of **PBench** and compute bipartite-matched mean IoU against ground-truth masks.

**`max_decode_steps_between_prefills`** — When the engine is used as a server handling concurrent requests, this parameter controls how many decode steps run before the scheduler checks for new prefills. Lower values improve **interactivity** (new requests start sooner), while higher values improve **throughput** (fewer context switches). The default of `8` is a good balance for interactive use.


In [ ]:
import json
import numpy as np
from falcon_perception.visualization_utils import decode_coco_rle, save_comparison_vis


def matched_iou(
    pred_masks: list[np.ndarray],
    gt_masks: list[np.ndarray],
    eval_size: int = 256,
) -> float:
    """Bipartite-matched mean IoU via vectorised matrix multiply."""
    if not pred_masks or not gt_masks:
        return 0.0

    from scipy.optimize import linear_sum_assignment

    def _flatten(masks):
        rows = []
        for m in masks:
            if m is None:
                continue
            if m.shape[0] != eval_size or m.shape[1] != eval_size:
                m = np.array(Image.fromarray((m > 0).astype(np.uint8) * 255).resize(
                    (eval_size, eval_size), Image.Resampling.NEAREST))
            rows.append((m > 0).ravel())
        return np.stack(rows).astype(np.float32) if rows else np.empty((0, eval_size * eval_size), dtype=np.float32)

    P = _flatten(pred_masks)
    G = _flatten(gt_masks)
    if P.shape[0] == 0 or G.shape[0] == 0:
        return 0.0

    inter = P @ G.T
    p_sum = P.sum(axis=1, keepdims=True)
    g_sum = G.sum(axis=1, keepdims=True)
    union = p_sum + g_sum.T - inter
    iou_matrix = np.where(union > 0, inter / union, 0.0)

    row_ind, col_ind = linear_sum_assignment(-iou_matrix)
    matched = iou_matrix[row_ind, col_ind]
    return float(matched.mean()) if len(matched) > 0 else 0.0


In [ ]:
BENCHMARK_SPLIT = "level_1"
BENCHMARK_LIMIT = 50

ds = stream_samples_from_hf_dataset("tiiuae/PBench", split=BENCHMARK_SPLIT, limit=BENCHMARK_LIMIT)

sequences = []
sample_data = []

for i, sample in enumerate(ds):
    expression = sample["expression"]
    prompt = build_prompt_for_task(str(expression), "segmentation")
    seq = Sequence(
        text=prompt,
        image=sample["image"],
        min_image_size=MIN_IMAGE_SIZE,
        max_image_size=MAX_IMAGE_SIZE,
        request_idx=i,
    )
    sequences.append(seq)
    sample_data.append(sample)

print(f"Built {len(sequences)} sequences from PBench/{BENCHMARK_SPLIT}")


In [ ]:
benchmark_sp = SamplingParams(
    stop_token_ids=stop_token_ids,
    coord_dedup_threshold=0.00,
    hr_upsample_ratio=8,
)

print("Benchmark warmup ...")
warmup_seq = sequences[0].copy()
warmup_seq.request_idx = 0
with cuda_timed(reset_peak_memory=False) as wt:
    engine.generate([warmup_seq], sampling_params=benchmark_sp)
print(f"Warmup done in {wt.elapsed:.1f}s")


In [ ]:
torch.cuda.reset_peak_memory_stats()
with cuda_timed() as timer:
    engine.generate(
        sequences,
        sampling_params=benchmark_sp,
        use_tqdm=True,
        print_stats=True,
    )

wall_time = timer.elapsed
print(f"\nBenchmark completed in {wall_time:.1f}s")


In [ ]:
all_ious = []
all_pred_counts = []
all_gt_counts = []
total_input_tokens = 0
total_output_tokens = 0
total_masks = 0

print(f"{'=' * 80}")
print("PER-SAMPLE RESULTS")
print("=" * 80)

for i, (seq, sample) in enumerate(zip(sequences, sample_data)):
    expression = sample["expression"]
    gt_rles = sample["masks"]

    gt_masks = []
    for rle in gt_rles:
        if isinstance(rle, dict):
            gt_masks.append(decode_coco_rle(rle))
        elif isinstance(rle, str):
            gt_masks.append(decode_coco_rle(json.loads(rle)))

    pred_masks = []
    for rle in seq.output_aux.masks_rle:
        if isinstance(rle, dict) and "counts" in rle:
            pred_masks.append(decode_coco_rle(rle))

    iou = matched_iou(pred_masks, gt_masks)
    decoded = tokenizer.decode(seq.output_ids)

    all_ious.append(iou)
    all_pred_counts.append(len(pred_masks))
    all_gt_counts.append(len(gt_masks))
    total_input_tokens += seq.input_length
    total_output_tokens += len(seq._output_ids)
    total_masks += len(seq.output_aux.masks_rle)

    print(f"\n[{i:3d}] expr={expression!r}")
    print(f"      GT masks: {len(gt_masks)}  |  Pred masks: {len(pred_masks)}  |  IoU: {iou:.3f}")
    print(f"      decoded: {decoded[:120]}{'...' if len(decoded) > 120 else ''}")


In [ ]:
total_tokens = total_input_tokens + total_output_tokens
total_tps = total_tokens / wall_time if wall_time > 0 else 0
images_per_sec = len(sequences) / wall_time if wall_time > 0 else 0

print(f"{'=' * 80}")
print("SUMMARY")
print("=" * 80)
print(f"  Dataset          : tiiuae/PBench / {BENCHMARK_SPLIT}")
print(f"  Samples          : {len(sequences)}")
print(f"  Wall time        : {wall_time:.1f}s")
print(f"  Total tok/s      : {total_tps:.1f}  (prefill + decode)")
print(f"  Images/s         : {images_per_sec:.2f}")
print(f"  Input tokens     : {total_input_tokens}")
print(f"  Output tokens    : {total_output_tokens}")
print(f"  Total masks      : {total_masks}")
print()
print(f"  Mean IoU         : {np.mean(all_ious):.4f}")
print(f"  Mean pred count  : {np.mean(all_pred_counts):.1f}")
print(f"  Mean GT count    : {np.mean(all_gt_counts):.1f}")
print(f"  Count match rate : {sum(1 for p, g in zip(all_pred_counts, all_gt_counts) if p == g)}/{len(sequences)}")
print(f"  Zero-pred samples: {sum(1 for p in all_pred_counts if p == 0)}/{len(sequences)}")

if torch.cuda.is_available():
    peak_alloc = torch.cuda.max_memory_allocated() / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
    print()
    print(f"  GPU              : {torch.cuda.get_device_name()}")
    print(f"  Peak VRAM alloc  : {peak_alloc:.2f} GiB")
    print(f"  Peak VRAM reserv : {peak_reserved:.2f} GiB")


In [ ]:
import os

vis_dir = "outputs_benchmark/vis"
os.makedirs(vis_dir, exist_ok=True)

for i, (seq, sample) in enumerate(zip(sequences, sample_data)):
    expression = sample["expression"]
    gt_rles = sample["masks"]
    pil_image = sample["image"].convert("RGB")

    gt_masks = []
    for rle in gt_rles:
        if isinstance(rle, dict):
            gt_masks.append(decode_coco_rle(rle))
        elif isinstance(rle, str):
            gt_masks.append(decode_coco_rle(json.loads(rle)))

    pred_masks = []
    for rle in seq.output_aux.masks_rle:
        if isinstance(rle, dict) and "counts" in rle:
            pred_masks.append(decode_coco_rle(rle))

    iou = all_ious[i]
    safe_expr = "".join(c if c.isalnum() or c in " _-" else "_" for c in expression)[:30].strip()
    vis_path = os.path.join(vis_dir, f"{i:04d}_iou{iou:.2f}_{safe_expr}.jpg")
    save_comparison_vis(pil_image, gt_masks, pred_masks, expression, iou, vis_path)

print(f"Saved {len(sequences)} comparison visualizations to {vis_dir}/")


---

## Summary

This notebook covered:

1. **Getting Started** — Loading Falcon Perception, building the paged inference engine, and running a warmup.
2. **Features** — Detection mode (faster with no upsampler), HR feature caching (skip upsampler on repeated images), and dense segmentation parameter tuning (`coord_dedup_threshold`, `hr_upsample_ratio`).
3. **Level Showcase** — Progressively harder queries (PBench levels 0–4) on a single image.
4. **Benchmark** — Quantitative evaluation on PBench with bipartite-matched mean IoU.

For more details, see the [model card on Hugging Face](https://huggingface.co/tiiuae/Falcon-Perception) and the scripts in this repository (`run_perception_single.py`, `run_perception_benchmark.py`).
